# 03 - Retrieval Demo & QA

Interactive demo notebook:
1. Text-only retrieval baseline
2. GNN-enhanced retrieval
3. Context expansion
4. Grounded answer generation
5. Side-by-side comparison


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import yaml
from sentence_transformers import CrossEncoder, SentenceTransformer

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'configs').exists() and (REPO_ROOT / '..' / 'configs').exists():
    REPO_ROOT = (REPO_ROOT / '..').resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.retrieval.retriever import BM25Retriever, TextRetriever, GNNRetriever, main_retrieval_test, print_results


def _resolve_path(*candidates):
    for candidate in candidates:
        if not candidate:
            continue
        p = Path(candidate)
        if p.is_absolute() and p.exists():
            return p
        local = REPO_ROOT / p
        if local.exists():
            return local
        parent = (REPO_ROOT / '..' / p).resolve()
        if parent.exists():
            return parent
    return None


def _load_faq_items():
    faq_candidates = [
        Path('data/processed/ai_act_faq_selected.json'),
        Path('data/processed/ai_act_faq.json'),
        Path('data/processed/ai_act_faq.csv'),
    ]
    faq_path = _resolve_path(*faq_candidates)
    if faq_path is None:
        raise FileNotFoundError(f'Could not find FAQ file. Tried: {[str(p) for p in faq_candidates]}')

    if faq_path.suffix.lower() == '.json':
        payload = json.loads(faq_path.read_text(encoding='utf-8'))
        if isinstance(payload, dict) and isinstance(payload.get('items'), list):
            items = payload['items']
        elif isinstance(payload, list):
            items = payload
        else:
            raise ValueError(f'Unsupported FAQ JSON format in {faq_path}')
    else:
        items = pd.read_csv(faq_path).to_dict(orient='records')

    queries = []
    for row in items:
        if not isinstance(row, dict):
            text = str(row).strip()
            if text:
                queries.append(text)
            continue
        for key in ['question', 'query', 'text', 'prompt', 'title']:
            if key in row and pd.notna(row[key]):
                text = str(row[key]).strip()
                if text:
                    queries.append(text)
                    break

    if not queries:
        raise ValueError(f'No queries found in {faq_path}')

    return items, queries, faq_path


config_path = _resolve_path('configs/config.yaml', '../configs/config.yaml')
if config_path is None:
    raise FileNotFoundError('Could not resolve configs/config.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

faq_items, queries, faq_path = _load_faq_items()
shared_encoder = SentenceTransformer(config['embedding']['model_name'])

cross_encoder = None
rerank_cfg = config.get('retrieval', {}).get('reranking', {})
if rerank_cfg.get('enabled', False):
    cross_encoder = CrossEncoder(rerank_cfg.get('model_name', 'cross-encoder/ms-marco-MiniLM-L-6-v2'))

nodes_path = _resolve_path(config['paths']['nodes_csv'], Path('..') / config['paths']['nodes_csv'])
edges_path = _resolve_path(config['paths']['edges_csv'], Path('..') / config['paths']['edges_csv'])
raw_embedding_path = _resolve_path(config['paths']['node_features'], Path('..') / config['paths']['node_features'])
gnn_embedding_path = _resolve_path(config['paths']['final_embeddings'], Path('..') / config['paths']['final_embeddings'])
graph_path = _resolve_path(config['paths']['graph_object'], Path('..') / config['paths']['graph_object'])

if None in [nodes_path, edges_path, raw_embedding_path, graph_path]:
    raise FileNotFoundError('Could not resolve one or more retrieval artifact paths from config')

nodes_df = pd.read_csv(nodes_path)
edges_df = pd.read_csv(edges_path)
graph_data = torch.load(graph_path, weights_only=False)
raw_features = np.load(raw_embedding_path)
raw_embeddings = np.asarray(raw_features[:, :384], dtype=np.float32)

print(f'Config:              {config_path}')
print(f'FAQ file:             {faq_path}')
print(f'Nodes path:           {nodes_path}')
print(f'Edges path:           {edges_path}')
print(f'Graph path:           {graph_path}')
print(f'Raw embedding path:   {raw_embedding_path}')
print(f'GNN embedding path:   {gnn_embedding_path}')
print(f'Nodes:                {len(nodes_df)}')
print(f'Edges:                {len(edges_df)}')
print(f'FAQ queries:          {len(queries)}')
print(f'Raw embeddings shape: {raw_embeddings.shape}')
print(f'Cross-encoder:       {rerank_cfg.get("model_name") if cross_encoder is not None else "disabled"}')


In [ ]:
# Load retrieval models
bm25_retriever = BM25Retriever(nodes_df=nodes_df)

text_retriever = TextRetriever(
    nodes_df=nodes_df,
    text_embeddings=raw_embeddings,
    model_name=config['embedding']['model_name'],
    encoder=shared_encoder,
)

gnn_retriever = None
gnn_embeddings = None
if gnn_embedding_path is not None and Path(gnn_embedding_path).exists():
    gnn_embeddings = np.load(gnn_embedding_path)
    if len(gnn_embeddings) == len(nodes_df):
        gnn_retriever = GNNRetriever(
            nodes_df=nodes_df,
            gnn_embeddings=gnn_embeddings,
            model_name=config['embedding']['model_name'],
            graph_data=graph_data,
            text_embeddings=raw_embeddings,
            encoder=shared_encoder,
        )

print(f'GNN available:       {gnn_retriever is not None}')
if gnn_embeddings is not None:
    print(f'GNN embedding shape: {gnn_embeddings.shape}')
print(f'BM25 enabled:        {config.get("retrieval", {}).get("bm25", {}).get("enabled", True)}')

demo_queries = queries[:5]


def _results_table(results, label, top_k=5):
    rows = []
    for r in results[:top_k]:
        rows.append({
            'method': label,
            'rank': r.get('rank'),
            'node_id': r.get('node_id'),
            'type': r.get('type'),
            'score': round(float(r.get('score', 0.0)), 4),
            'cross': round(float(r.get('cross_score', 0.0)), 4),
            'expanded': bool(r.get('expanded', False)),
            'preview': str(r.get('text', ''))[:90].replace('\n', ' '),
        })
    return pd.DataFrame(rows)


def plot_retrieved_subgraph(results, title_suffix=''):
    if graph_data is None or not hasattr(graph_data, 'edge_index'):
        return

    node_ids = [r['node_id'] for r in results]
    if not node_ids:
        return

    id_to_idx = {str(node_id): idx for idx, node_id in enumerate(nodes_df['node_id'].astype(str).tolist())}
    selected = {node_id for node_id in node_ids if node_id in id_to_idx}
    if not selected:
        return

    edge_index = graph_data.edge_index
    if torch.is_tensor(edge_index):
        edge_index = edge_index.detach().cpu().numpy()

    H = nx.DiGraph()
    seed_nodes = {r['node_id'] for r in results if not r.get('expanded', False)}
    expanded_nodes = {r['node_id'] for r in results if r.get('expanded', False)}

    for row in results:
        node_id = row['node_id']
        H.add_node(node_id, node_type=row.get('type', 'unknown'), expanded=row.get('expanded', False), score=row.get('score', 0.0))

    for src_idx, dst_idx in edge_index.T:
        src_id = str(nodes_df.iloc[int(src_idx)]['node_id'])
        dst_id = str(nodes_df.iloc[int(dst_idx)]['node_id'])
        if src_id in selected and dst_id in selected:
            H.add_edge(src_id, dst_id)

    if H.number_of_nodes() == 0:
        return

    fig, ax = plt.subplots(figsize=(9, 6))
    pos = nx.spring_layout(H, seed=42, k=1.1 / max(np.sqrt(H.number_of_nodes()), 1.0))
    node_colors = []
    node_sizes = []
    for node_id in H.nodes():
        if node_id in seed_nodes:
            node_colors.append('#E53935')
            node_sizes.append(480)
        elif node_id in expanded_nodes:
            node_colors.append('#1E88E5')
            node_sizes.append(320)
        else:
            node_colors.append('#9E9E9E')
            node_sizes.append(280)

    nx.draw_networkx_edges(H, pos, alpha=0.30, arrows=True, arrowsize=10, ax=ax)
    nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(H, pos, labels={n: n for n in H.nodes()}, font_size=7, ax=ax)
    ax.set_title(f'Hybrid Retrieved Subgraph{title_suffix}', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def run_demo(query, top_k=None, expand=False):
    if top_k is None:
        top_k = config.get('retrieval', {}).get('top_k', 5)

    text_results = text_retriever.retrieve(query, top_k=top_k)
    bm25_results = bm25_retriever.retrieve(query, top_k=top_k)
    hybrid_results = main_retrieval_test(
        query=query,
        nodes_df=nodes_df,
        text_embeddings=raw_embeddings,
        model_name=config['embedding']['model_name'],
        top_k=top_k,
        seed_k=top_k,
        graph_data=graph_data,
        gnn_embeddings=gnn_embeddings,
        expand_hops=config.get('retrieval', {}).get('context_expansion', {}).get('hops', 1),
        max_expand=config.get('retrieval', {}).get('context_expansion', {}).get('max_neighbors', 10),
        cross_encoder=cross_encoder,
        rerank_weight=config.get('retrieval', {}).get('reranking', {}).get('weight', 0.4),
        shared_encoder=shared_encoder,
    )

    print(f"\nQuery: {query}")
    summary = pd.concat([
        _results_table(text_results, 'dense', top_k=top_k),
        _results_table(bm25_results, 'bm25', top_k=top_k),
        _results_table(hybrid_results, 'hybrid', top_k=top_k),
    ], ignore_index=True)
    print(summary.to_string(index=False))

    if expand and gnn_retriever is not None:
        plot_retrieved_subgraph(hybrid_results, title_suffix=f'\nQuery: {query[:80]}')

    return {
        'dense': text_results,
        'bm25': bm25_results,
        'hybrid': hybrid_results,
    }


all_demo_results = []
for query in demo_queries:
    all_demo_results.append(run_demo(query, top_k=5, expand=config.get('retrieval', {}).get('context_expansion', {}).get('enabled', False)))
